In [ ]:
import pandas as pd


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

filename = next(iter(uploaded))

costs_raw = pd.read_excel(filename, sheet_name="Costs")

print("rows, columns", costs_raw.shape)

display(costs_raw.head())

In [ ]:
# Keep the original data unchanged.
costs = costs_raw.copy()

print("Original shape:", costs_raw.shape)
print("Working shape:", costs.shape)

In [ ]:
# Count missing values in each column.
missing_counts = costs.isna().sum()

print("Missing values by column:")
display(missing_counts)

In [ ]:
costs.duplicated()

In [ ]:
costs["Cost_ID"].duplicated()

In [ ]:
# Count records for each cost category.
cost_type_counts = costs["Cost_Type"].value_counts(dropna=False)

print("Records by cost type:")
display(cost_type_counts)

In [ ]:
# Inspect the original route labels.
print("Original route label count:", costs["Route_ID"].nunique())

display(costs[["Route_ID"]].head(10))

In [ ]:
# Preserve the original route labels.
costs["Route_ID_Original"] = costs["Route_ID"]

# Convert route labels to uppercase and remove separators.
route_compact = (
    costs["Route_ID_Original"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z]", "", regex=True))

display(
    pd.DataFrame({
        "OriginalRoute": costs["Route_ID_Original"],
        "CompactRoute": route_compact
    }).head(10))


In [ ]:
# Validate the six-letter route format.
valid_route = route_compact.str.fullmatch(r"[A-Z]{6}", na=False)

invalid_route_count = (~valid_route).sum()

print("Invalid route labels:", invalid_route_count)

if invalid_route_count > 0:
    display(
        costs.loc[
            ~valid_route,
            ["Cost_ID", "Route_ID_Original"]
        ].head(20))
    raise ValueError("Invalid route labels found. Review them before continuing.")

In [ ]:
# Standardize routes as departure-arrival.
costs["Route_ID"] = (
    route_compact.str[:3]
    + "-"
    + route_compact.str[3:])

print("Original route label count:",
      costs["Route_ID_Original"].nunique())

print("Standardized directional route count:",
      costs["Route_ID"].nunique())

display(
    costs[
        ["Cost_ID", "Route_ID_Original", "Route_ID"]
    ].head(10))

In [ ]:
# Confirm that route cleaning preserved all records and amounts.
assert len(costs) == len(costs_raw), "The row count changed."
assert costs["Amount"].equals(costs_raw["Amount"]), "The amounts changed."

print("Row count preserved:", len(costs))
print("Amount values unchanged: True")
print("Working columns:", costs.columns.tolist())

In [ ]:
# Convert dates and amounts to calculation-ready types.
parsed_dates = pd.to_datetime(costs["Cost_Date"], errors="coerce")
parsed_amounts = pd.to_numeric(costs["Amount"], errors="coerce")

print("Missing or invalid dates:", parsed_dates.isna().sum())
print("Missing or invalid amounts:", parsed_amounts.isna().sum())

assert parsed_dates.notna().all(), "Review missing or invalid dates."
assert parsed_amounts.notna().all(), "Review missing or invalid amounts."

costs["Cost_Date"] = parsed_dates
costs["Amount"] = parsed_amounts

print("\nDate range:")
print("Start:", costs["Cost_Date"].min())
print("End:", costs["Cost_Date"].max())

print("\nAmount range:")
print("Minimum:", costs["Amount"].min())
print("Maximum:", costs["Amount"].max())

print("\nNegative amounts:", (costs["Amount"] < 0).sum())
print("Zero amounts:", (costs["Amount"] == 0).sum())

In [ ]:
# Remove surrounding spaces from cost category labels.
costs["Cost_Type"] = costs["Cost_Type"].astype("string").str.strip()

expected_types = {"Crew", "Food", "Fuel", "Maintenance"}

print("Cost category counts:")
display(costs["Cost_Type"].value_counts(dropna=False))

assert costs["Cost_Type"].notna().all(), "Missing cost categories found."
assert set(costs["Cost_Type"].unique()) == expected_types, (
    "Unexpected cost categories found."
)

assert costs["Cost_ID"].notna().all(), "Missing Cost_ID values found."
assert costs["Cost_ID"].is_unique, "Duplicate Cost_ID values found."

print("Cost category and identifier checks passed.")

In [ ]:
# Validate each route-time combination before treating it as a flight.
flight_keys = ["Route_ID", "Cost_Date"]

flight_checks = (
    costs.groupby(flight_keys)
    .agg(
        RecordCount=("Cost_ID", "size"),
        CostTypeCount=("Cost_Type", "nunique"))
    .reset_index())

invalid_flights = flight_checks.loc[
    (flight_checks["RecordCount"] != 4)
    | (flight_checks["CostTypeCount"] != 4)]

print("Route-time combinations:", len(flight_checks))
print("Invalid combinations:", len(invalid_flights))

if not invalid_flights.empty:
    display(invalid_flights.head(20))
    raise ValueError("Review incomplete or duplicated flight cost records.")

print("All combinations contain four records and four cost categories.")

In [ ]:
# Reshape validated cost records into one row per inferred flight.
cost_columns = ["Crew", "Food", "Fuel", "Maintenance"]

flight_costs = (
    costs.pivot(
        index=flight_keys,
        columns="Cost_Type",
        values="Amount")
    .reset_index())

flight_costs.columns.name = None

# Calculate the total operating cost for each inferred flight.
flight_costs["OperatingCost"] = (
    flight_costs[cost_columns].sum(axis=1))

# Display flights in chronological order.
flight_costs = (
    flight_costs
    .sort_values(["Cost_Date", "Route_ID"])
    .reset_index(drop=True))

print("Inferred flight count:", len(flight_costs))
display(flight_costs.head())

In [ ]:
import numpy as np

# Reconcile totals before and after reshaping.
source_total = costs["Amount"].sum()
flight_total = flight_costs["OperatingCost"].sum()
difference = flight_total - source_total

print(f"Source operating cost: {source_total:,.2f}")
print(f"Flight table operating cost: {flight_total:,.2f}")
print(f"Difference: {difference:,.6f}")

assert np.isclose(
    source_total,
    flight_total,
    rtol=0,
    atol=0.01
), "Operating cost totals do not reconcile."

assert len(flight_costs) * 4 == len(costs), (
    "The record count does not match four records per flight."
)

print("Reconciliation checks passed.")

In [ ]:
# Add calendar month and departure airport.
flight_costs["Month"] = (
    flight_costs["Cost_Date"].dt.to_period("M").astype(str)
)

flight_costs["DepartureAirport"] = (
    flight_costs["Route_ID"].str[:3]
)

# The fiscal year starts on September 1.
fiscal_start_year = (
    flight_costs["Cost_Date"].dt.year
    - (flight_costs["Cost_Date"].dt.month < 9).astype(int)
)

flight_costs["FY"] = (
    "FY"
    + fiscal_start_year.astype(str)
    + "-"
    + ((fiscal_start_year + 1) % 100).astype(str).str.zfill(2)
)

display(
    flight_costs[
        ["Cost_Date", "Month", "FY", "Route_ID", "DepartureAirport"]
    ].head()
)

print("Fiscal year flight counts:")
display(flight_costs["FY"].value_counts().sort_index())

print("First month:", flight_costs["Month"].min())
print("Last month:", flight_costs["Month"].max())
print("Observed month count:", flight_costs["Month"].nunique())

In [ ]:
# Summarize costs and flight counts by month and directional route.
route_monthly_costs = (
    flight_costs.groupby(
        ["Month", "FY", "Route_ID", "DepartureAirport"],
        as_index=False
    )
    .agg(
        CrewCost=("Crew", "sum"),
        FoodCost=("Food", "sum"),
        FuelCost=("Fuel", "sum"),
        MaintenanceCost=("Maintenance", "sum"),
        OperatingCost=("OperatingCost", "sum"),
        FlightCount=("OperatingCost", "size")))

# Calculate average operating cost per flight.
route_monthly_costs["CostPerFlight"] = (
    route_monthly_costs["OperatingCost"]
    / route_monthly_costs["FlightCount"])

route_monthly_costs = (
    route_monthly_costs
    .sort_values(["Month", "Route_ID"])
    .reset_index(drop=True))

display(route_monthly_costs.head(10).round(2))

print("Summary rows:", len(route_monthly_costs))

In [ ]:
# Inspect monthly results for one directional route.
example_route = route_monthly_costs.loc[
    route_monthly_costs["Route_ID"] == "YVR-YLW",
    [
        "Month",
        "CrewCost",
        "FoodCost",
        "FuelCost",
        "MaintenanceCost",
        "OperatingCost",
        "FlightCount",
        "CostPerFlight"]]

display(example_route.head(6).round(2))


In [ ]:
# Summarize company operating costs by fiscal year.
annual_costs = (
    flight_costs.groupby("FY", as_index=False)
    .agg(
        CrewCost=("Crew", "sum"),
        FoodCost=("Food", "sum"),
        FuelCost=("Fuel", "sum"),
        MaintenanceCost=("Maintenance", "sum"),
        OperatingCost=("OperatingCost", "sum"),
        FlightCount=("OperatingCost", "size")
    )
    .sort_values("FY")
    .reset_index(drop=True)
)

annual_costs["CostPerFlight"] = (
    annual_costs["OperatingCost"]
    / annual_costs["FlightCount"]
)

display(annual_costs.round(2))

In [ ]:
# Verify that aggregation preserved total costs and flight counts.
assert np.isclose(
    route_monthly_costs["OperatingCost"].sum(),
    flight_costs["OperatingCost"].sum(),
    rtol=0,
    atol=0.01
), "Monthly route cost totals do not reconcile."

assert (
    route_monthly_costs["FlightCount"].sum() == len(flight_costs)
), "Monthly route flight counts do not reconcile."

assert np.isclose(
    annual_costs["OperatingCost"].sum(),
    flight_costs["OperatingCost"].sum(),
    rtol=0,
    atol=0.01
), "Annual cost totals do not reconcile."

assert (
    annual_costs["FlightCount"].sum() == len(flight_costs)
), "Annual flight counts do not reconcile."

assert not route_monthly_costs.duplicated(
    ["Month", "Route_ID"]
).any(), "Duplicate month-route combinations found."

print("All summary checks passed.")

In [ ]:
# Compare the two fiscal years.
annual_indexed = annual_costs.set_index("FY")

previous = annual_indexed.loc["FY2024-25"]
current = annual_indexed.loc["FY2025-26"]

metrics = [
    "CrewCost",
    "FoodCost",
    "FuelCost",
    "MaintenanceCost",
    "OperatingCost",
    "FlightCount",
    "CostPerFlight"
]

annual_comparison = pd.DataFrame({
    "PreviousYear": previous[metrics],
    "CurrentYear": current[metrics]
})

annual_comparison["Change"] = (
    annual_comparison["CurrentYear"]
    - annual_comparison["PreviousYear"]
)

annual_comparison["GrowthPct"] = (
    annual_comparison["Change"]
    / annual_comparison["PreviousYear"]
    * 100
)

annual_comparison.index.name = "Metric"

display(annual_comparison.round(2))

In [ ]:
# Summarize company operating costs by month.
monthly_costs = (
    flight_costs.groupby(["Month", "FY"], as_index=False)
    .agg(
        CrewCost=("Crew", "sum"),
        FoodCost=("Food", "sum"),
        FuelCost=("Fuel", "sum"),
        MaintenanceCost=("Maintenance", "sum"),
        OperatingCost=("OperatingCost", "sum"),
        FlightCount=("OperatingCost", "size")
    )
    .sort_values("Month")
    .reset_index(drop=True)
)

monthly_costs["CostPerFlight"] = (
    monthly_costs["OperatingCost"]
    / monthly_costs["FlightCount"]
)

assert np.isclose(
    monthly_costs["OperatingCost"].sum(),
    flight_costs["OperatingCost"].sum(),
    rtol=0,
    atol=0.01
), "Monthly operating costs do not reconcile."

assert monthly_costs["FlightCount"].sum() == len(flight_costs), (
    "Monthly flight counts do not reconcile."
)

print("Observed months:", len(monthly_costs))
display(monthly_costs.round(2))

In [ ]:
# Display floating-point values with separators and two decimal places.
pd.options.display.float_format = "{:,.2f}".format

display(annual_comparison)

In [ ]:
import matplotlib.pyplot as plt

# Prepare monthly data in chronological order.
plot_data = monthly_costs.sort_values("Month").copy()
month_dates = pd.to_datetime(plot_data["Month"], format="%Y-%m")

# Create three panels with a shared time axis.
fig, axes = plt.subplots(
    3,
    1,
    figsize=(12, 10),
    sharex=True
)

# Panel 1: total monthly operating costs.
axes[0].plot(
    month_dates,
    plot_data["OperatingCost"] / 1_000_000,
    marker="o",
    color="#2563EB"
)
axes[0].set_title("Monthly Operating Costs")
axes[0].set_ylabel("Cost (millions)")

# Panel 2: monthly inferred flight counts.
axes[1].plot(
    month_dates,
    plot_data["FlightCount"],
    marker="o",
    color="#059669"
)
axes[1].set_title("Monthly Inferred Flight Counts")
axes[1].set_ylabel("Flights")

# Panel 3: average operating cost per flight.
axes[2].plot(
    month_dates,
    plot_data["CostPerFlight"],
    marker="o",
    color="#D97706"
)
axes[2].set_title("Average Operating Cost per Flight")
axes[2].set_ylabel("Cost per flight")
axes[2].set_xlabel("Month")

# Mark the start of the second fiscal year.
for ax in axes:
    ax.axvline(
        pd.Timestamp("2025-09-01"),
        color="gray",
        linestyle="--",
        linewidth=1
    )
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(bottom=0)

axes[2].set_xticks(month_dates)
axes[2].set_xticklabels(
    plot_data["Month"],
    rotation=45,
    ha="right"
)

fig.suptitle(
    "Operating Cost, Flight Volume, and Average Cost",
    fontsize=15
)

fig.tight_layout()

# Save the chart in the current runtime.
fig.savefig(
    "monthly_operating_cost_trends.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Select the four operating cost categories.
category_names = [
    "CrewCost",
    "FoodCost",
    "FuelCost",
    "MaintenanceCost"
]

cost_increase_breakdown = (
    annual_comparison.loc[category_names, ["Change", "GrowthPct"]]
    .copy()
)

# Calculate each category's share of the total cost increase.
total_cost_increase = annual_comparison.loc[
    "OperatingCost", "Change"
]

assert total_cost_increase > 0, "Total operating costs did not increase."

cost_increase_breakdown["ShareOfIncreasePct"] = (
    cost_increase_breakdown["Change"]
    / total_cost_increase
    * 100
)

cost_increase_breakdown = (
    cost_increase_breakdown
    .sort_values("Change", ascending=False)
)

# Confirm that the category increases equal the total increase.
assert np.isclose(
    cost_increase_breakdown["Change"].sum(),
    total_cost_increase,
    rtol=0,
    atol=0.01
), "Category increases do not reconcile."

display(cost_increase_breakdown.round(2))

In [ ]:
# Order categories for a horizontal bar chart.
chart_data = cost_increase_breakdown.sort_values("Change").copy()

category_labels = {
    "CrewCost": "Crew",
    "FoodCost": "Food",
    "FuelCost": "Fuel",
    "MaintenanceCost": "Maintenance"
}

labels = [
    category_labels[name]
    for name in chart_data.index
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    labels,
    chart_data["Change"] / 1_000_000,
    color="#2563EB"
)

# Annotate the increase amount and contribution share.
for bar, amount, share in zip(
    bars,
    chart_data["Change"],
    chart_data["ShareOfIncreasePct"]
):
    ax.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{amount / 1_000_000:.2f}m ({share:.1f}%)",
        va="center"
    )

ax.set_title(
    "Contribution to Operating Cost Increase\n"
    "FY2025-26 vs FY2024-25"
)
ax.set_xlabel("Increase in operating cost (millions)")
ax.set_xlim(
    0,
    chart_data["Change"].max() / 1_000_000 * 1.35
)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()

fig.savefig(
    "operating_cost_increase_by_category.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Summarize operating costs by fiscal year and directional route.
route_annual_costs = (
    flight_costs.groupby(["FY", "Route_ID"], as_index=False)
    .agg(
        CrewCost=("Crew", "sum"),
        FoodCost=("Food", "sum"),
        FuelCost=("Fuel", "sum"),
        MaintenanceCost=("Maintenance", "sum"),
        OperatingCost=("OperatingCost", "sum"),
        FlightCount=("OperatingCost", "size"),
        ActiveMonths=("Month", "nunique")
    )
)

route_annual_costs["CostPerFlight"] = (
    route_annual_costs["OperatingCost"]
    / route_annual_costs["FlightCount"]
)

route_annual_costs = (
    route_annual_costs
    .sort_values(["FY", "Route_ID"])
    .reset_index(drop=True)
)

display(route_annual_costs.head(10))

# Reconcile route totals with company totals.
route_fy_totals = (
    route_annual_costs.groupby("FY")[
        ["OperatingCost", "FlightCount"]
    ]
    .sum()
    .sort_index()
)

company_fy_totals = (
    annual_costs.set_index("FY")[
        ["OperatingCost", "FlightCount"]
    ]
    .sort_index()
)

assert route_fy_totals.index.equals(company_fy_totals.index)

assert np.allclose(
    route_fy_totals["OperatingCost"],
    company_fy_totals["OperatingCost"],
    rtol=0,
    atol=0.01
), "Route costs do not reconcile with company costs."

assert route_fy_totals["FlightCount"].equals(
    company_fy_totals["FlightCount"]
), "Route flight counts do not reconcile."

print("Route annual reconciliation passed.")

In [ ]:
# Select the metrics needed for year-over-year comparison.
comparison_columns = [
    "Route_ID",
    "OperatingCost",
    "FlightCount",
    "CostPerFlight",
    "ActiveMonths"
]

previous_routes = route_annual_costs.loc[
    route_annual_costs["FY"] == "FY2024-25",
    comparison_columns
].copy()

current_routes = route_annual_costs.loc[
    route_annual_costs["FY"] == "FY2025-26",
    comparison_columns
].copy()

# Keep all routes, including those observed in only one fiscal year.
route_comparison = previous_routes.merge(
    current_routes,
    on="Route_ID",
    how="outer",
    suffixes=("_Previous", "_Current"),
    validate="one_to_one",
    indicator=True
)

print("Route coverage across fiscal years:")
display(route_comparison["_merge"].value_counts())

# Treat absent records as zero observed costs and flights.
# Average cost per flight remains undefined when no flights are observed.
for metric in ["OperatingCost", "FlightCount", "ActiveMonths"]:
    columns = [
        f"{metric}_Previous",
        f"{metric}_Current"
    ]
    route_comparison[columns] = (
        route_comparison[columns].fillna(0)
    )

for metric in ["FlightCount", "ActiveMonths"]:
    for period in ["Previous", "Current"]:
        column = f"{metric}_{period}"
        route_comparison[column] = (
            route_comparison[column].astype(int)
        )

display(route_comparison.head())

In [ ]:
# Calculate absolute changes.
route_comparison["CostChange"] = (
    route_comparison["OperatingCost_Current"]
    - route_comparison["OperatingCost_Previous"]
)

route_comparison["FlightChange"] = (
    route_comparison["FlightCount_Current"]
    - route_comparison["FlightCount_Previous"]
)

route_comparison["CostPerFlightChange"] = (
    route_comparison["CostPerFlight_Current"]
    - route_comparison["CostPerFlight_Previous"]
)

# Calculate growth rates only when the previous value is positive.
for metric, output_column in [
    ("OperatingCost", "CostGrowthPct"),
    ("FlightCount", "FlightGrowthPct"),
    ("CostPerFlight", "CostPerFlightGrowthPct")
]:
    previous_values = route_comparison[f"{metric}_Previous"]
    current_values = route_comparison[f"{metric}_Current"]

    valid_base = previous_values.where(previous_values > 0)

    route_comparison[output_column] = (
        (current_values - previous_values)
        / valid_base
        * 100
    )

# Reconcile the total route cost change.
assert np.isclose(
    route_comparison["CostChange"].sum(),
    total_cost_increase,
    rtol=0,
    atol=0.01
), "Route cost changes do not reconcile."

route_comparison = (
    route_comparison
    .sort_values("CostChange", ascending=False)
    .reset_index(drop=True)
)

result_columns = [
    "Route_ID",
    "ActiveMonths_Previous",
    "ActiveMonths_Current",
    "OperatingCost_Previous",
    "OperatingCost_Current",
    "CostChange",
    "FlightCount_Previous",
    "FlightCount_Current",
    "FlightChange",
    "CostPerFlight_Previous",
    "CostPerFlight_Current",
    "CostPerFlightGrowthPct"
]

display(route_comparison[result_columns].head(10).round(2))

print("Total route cost increase:",
      f"{route_comparison['CostChange'].sum():,.2f}")

In [ ]:
# Separate routes observed in both fiscal years from other routes.
has_both_years = (
    (route_comparison["FlightCount_Previous"] > 0)
    & (route_comparison["FlightCount_Current"] > 0)
)

route_decomposition = route_comparison.loc[
    has_both_years
].copy()

# Step 1: change flight counts while holding previous average costs.
route_decomposition["FlightCountEffect"] = (
    route_decomposition["FlightCount_Current"]
    - route_decomposition["FlightCount_Previous"]
) * route_decomposition["CostPerFlight_Previous"]

# Step 2: change average costs using current flight counts.
route_decomposition["AverageCostEffect"] = (
    route_decomposition["FlightCount_Current"]
    * (
        route_decomposition["CostPerFlight_Current"]
        - route_decomposition["CostPerFlight_Previous"]
    )
)

route_decomposition["ExplainedChange"] = (
    route_decomposition["FlightCountEffect"]
    + route_decomposition["AverageCostEffect"]
)

assert np.allclose(
    route_decomposition["ExplainedChange"],
    route_decomposition["CostChange"],
    rtol=0,
    atol=0.01
), "Route cost decomposition does not reconcile."

# Retain changes for routes without observations in both years.
other_routes = route_comparison.loc[~has_both_years].copy()

explained_total = (
    route_decomposition["ExplainedChange"].sum()
    + other_routes["CostChange"].sum()
)

assert np.isclose(
    explained_total,
    total_cost_increase,
    rtol=0,
    atol=0.01
), "Company cost decomposition does not reconcile."

print("Routes observed in both years:", len(route_decomposition))
print("Other routes:", len(other_routes))
print("Decomposition checks passed.")

display(
    route_decomposition[
        [
            "Route_ID",
            "ActiveMonths_Previous",
            "ActiveMonths_Current",
            "CostChange",
            "FlightCountEffect",
            "AverageCostEffect"
        ]
    ]
    .sort_values("CostChange", ascending=False)
    .head(10)
    .round(2)
)

In [ ]:
# Summarize the accounting decomposition across routes.
decomposition_summary = pd.DataFrame({
    "Component": [
        "Flight count changes at previous route costs",
        "Average cost changes at current flight counts",
        "Routes without observations in both years"
    ],
    "Amount": [
        route_decomposition["FlightCountEffect"].sum(),
        route_decomposition["AverageCostEffect"].sum(),
        other_routes["CostChange"].sum()
    ]
})

decomposition_summary["ShareOfNetIncreasePct"] = (
    decomposition_summary["Amount"]
    / total_cost_increase
    * 100
)

display(decomposition_summary.round(2))

print("Total operating cost increase:",
      f"{total_cost_increase:,.2f}")

In [ ]:
# Select the ten comparable routes with the largest cost increases.
top_routes = (
    route_decomposition
    .nlargest(10, "CostChange")
    .sort_values("CostChange")
    .set_index("Route_ID")
)

plot_values = top_routes[
    ["FlightCountEffect", "AverageCostEffect"]
].div(1_000_000)

plot_values = plot_values.rename(columns={
    "FlightCountEffect": "Flight count effect",
    "AverageCostEffect": "Average cost effect"
})

ax = plot_values.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 7),
    color=["#2563EB", "#F59E0B"]
)

ax.set_title(
    "Operating Cost Increase by Directional Route\n"
    "FY2025-26 vs FY2024-25"
)
ax.set_xlabel("Cost change (millions)")
ax.set_ylabel("Directional route")
ax.axvline(0, color="black", linewidth=0.8)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
ax.legend(title=None, loc="lower right")

fig = ax.get_figure()
fig.tight_layout()

fig.savefig(
    "route_cost_increase_decomposition.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    "route_cost_increase_decomposition.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Count departures by month and airport for gate-rent allocation.
airport_monthly_departures = (
    flight_costs.groupby(
        ["Month", "FY", "DepartureAirport"],
        as_index=False
    )
    .agg(
        DepartureFlightCount=("OperatingCost", "size")
    )
    .sort_values(["Month", "DepartureAirport"])
    .reset_index(drop=True)
)

assert (
    airport_monthly_departures["DepartureFlightCount"].sum()
    == len(flight_costs)
), "Airport departure counts do not reconcile."

display(airport_monthly_departures.head(10))

print("Airport departure count check passed.")

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Create the export folder.
output_dir = Path("operating_cost_outputs")
output_dir.mkdir(exist_ok=True)

# Export analysis tables.
export_tables = {
    "flight_costs.csv": flight_costs,
    "route_monthly_costs.csv": route_monthly_costs,
    "airport_monthly_departures.csv": airport_monthly_departures,
    "monthly_costs.csv": monthly_costs,
    "annual_costs.csv": annual_costs,
    "annual_comparison.csv": annual_comparison.reset_index(),
    "route_annual_costs.csv": route_annual_costs,
    "route_comparison.csv": route_comparison,
    "cost_increase_breakdown.csv": cost_increase_breakdown.reset_index(),
    "route_decomposition.csv": route_decomposition,
    "decomposition_summary.csv": decomposition_summary
}

for filename, table in export_tables.items():
    table.to_csv(
        output_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Copy the charts produced earlier.
chart_names = [
    "monthly_operating_cost_trends.png",
    "operating_cost_increase_by_category.png",
    "route_cost_increase_decomposition.png",
    "route_cost_increase_decomposition.svg"
]

for filename in chart_names:
    source = Path(filename)
    assert source.exists(), f"Missing chart: {filename}"
    shutil.copy2(source, output_dir / filename)

# Record the main assumptions and findings.
readme_text = """Operating Cost Analysis — First Byte 2026

Scope
- Four operating cost categories: Crew, Food, Fuel, Maintenance.
- Gate rent is excluded.
- Flights are inferred from normalized Route_ID + Cost_Date.
- Each inferred flight was validated to have four records and four categories.
- Fiscal years run from September 1 to August 31.
- Month uses YYYY-MM; Route_ID preserves direction.
- No-flight route-month combinations are omitted from the cost summary.

Validated totals
- Inferred flights: 28,801.
- Total operating costs: 431,362,040.24.
- FY2024-25 operating costs: 192,985,563.99; flights: 13,072.
- FY2025-26 operating costs: 238,376,476.25; flights: 15,729.

Findings
- Operating costs increased by 45,390,912.26.
- Fuel and crew contributed 68.68% of the net cost increase.
- Route-level sequential decomposition:
  flight count effect = (N1 - N0) * c0;
  average cost effect = N1 * (c1 - c0).
- Flight count effect: 30,617,927.95.
- Average cost effect: 14,772,984.31.

Limitations
- The decomposition is an accounting identity, not causal evidence.
- Flight count changes can include additional active months.
- Average cost changes can include seasonal and operational differences.
- Operating costs alone do not establish route profitability.
- Revenue processing dates do not directly identify flight dates.

Handoff
- Revenue analysis: merge route_monthly_costs.csv on Month + Route_ID.
- Gate-rent analysis: use airport_monthly_departures.csv and
  route_monthly_costs.csv for departure-based allocation.
- Preserve airport rent even when no departures are observed.
"""

(output_dir / "README.txt").write_text(
    readme_text,
    encoding="utf-8"
)

# Package only the files created for this handoff.
archive_path = Path("FirstByte_Operating_Cost_Outputs.zip")

export_names = (
    list(export_tables.keys())
    + chart_names
    + ["README.txt"]
)

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in export_names:
        archive.write(
            output_dir / filename,
            arcname=f"operating_cost_outputs/{filename}"
        )

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None, "Archive validation failed."

print("Exported tables:", len(export_tables))
print("Exported charts:", len(chart_names))
print("Archive:", archive_path)

In [ ]:
from google.colab import files

files.download("FirstByte_Operating_Cost_Outputs.zip")